In [1]:
import json
import os, statistics, math

def extract_step_to_acc(path: str):
    """
    Read a raw metrics file and return {step: test_acc} from even-numbered lines only.
    - 1-based line numbering (keep only even lines)
    - JSON parse; keep only split == 'test'
    - Map: step -> acc (latest occurrence wins if duplicated)
    """
    result = {}
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):  # 1-based
            line = line.strip()
            if not line or (i % 2 != 0):  # skip odd lines
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            if obj.get("split") != "test":
                continue
            step = obj.get("step")
            acc = obj.get("acc")
            if step is not None and acc is not None:
                result[step] = acc  # latest one wins
    return result

In [2]:
def summarize_by_step(root='.', start=1, end=53, prefix='client_', ext='.raw',
                      variance='population', step_keys=None, require_all=False):
    """
    Aggregate across clients PER STEP and return TWO dictionaries:
      - mean_by_step: {step: mean_test_acc_over_clients}
      - var_by_step:  {step: variance_test_acc_over_clients}

    Parameters:
      - variance: 'population' (statistics.pvariance) | 'sample' (statistics.variance)
      - step_keys: optional iterable of steps to enforce (e.g., [0,10,20,...,100]).
                   If None, uses the union of steps observed across clients.
      - require_all: if True, only include a step if ALL clients have a value for it.
                     If False, compute from available values.

    Notes:
      - Uses extract_step_to_acc(path) from earlier cell (even-line, split=='test').
      - Missing files or missing steps are ignored per 'require_all' policy.
    """
    # Collect per-step lists of accuracies across clients
    accs_by_step = {}
    client_count = 0
    for i in range(start, end + 1):
        path = os.path.join(root, f"{prefix}{i:03d}{ext}")
        try:
            d = extract_step_to_acc(path)
        except FileNotFoundError:
            # Missing client file; skip
            continue
        client_count += 1
        for step, acc in d.items():
            accs_by_step.setdefault(step, []).append(acc)

    # Decide which steps to include
    if step_keys is None:
        steps = sorted(accs_by_step.keys())
    else:
        steps = list(step_keys)

    mean_by_step = {}
    var_by_step  = {}
    for step in steps:
        vals = accs_by_step.get(step, [])
        if not vals:
            continue  # no data at this step
        if require_all and len(vals) != client_count:
            # Skip this step because not all clients provided it
            continue
        m = sum(vals) / len(vals)
        if variance == 'population':
            v = statistics.pvariance(vals)
        elif variance == 'sample':
            v = statistics.variance(vals) if len(vals) > 1 else float('nan')
        else:
            raise ValueError("variance must be 'population' or 'sample'")
        mean_by_step[step] = m
        var_by_step[step]  = v
    return mean_by_step, var_by_step

In [3]:
mean_dict, var_dict = summarize_by_step(root='.', start=1, end=53)
print(mean_dict)
print(var_dict)

{0: 0.8169999999999998, 10: 0.8099999999999996, 20: 0.8210000000000001, 30: 0.8254999999999999, 40: 0.8170000000000002, 50: 0.8239999999999998, 60: 0.8285, 70: 0.8320000000000004, 80: 0.8325000000000001, 90: 0.834, 100: 0.8405000000000002, 110: 0.8384999999999998, 120: 0.8305, 130: 0.8360000000000001, 140: 0.8390000000000001, 150: 0.8480000000000002, 160: 0.8530000000000003, 170: 0.8485000000000001, 180: 0.8434999999999999, 190: 0.844, 200: 0.8410000000000002, 210: 0.8424999999999999, 220: 0.8455000000000001, 230: 0.8530000000000001, 240: 0.8525000000000003, 250: 0.8485000000000001, 260: 0.8497435897435895, 270: 0.8463157894736841, 280: 0.8500000000000003, 290: 0.8459459459459459, 300: 0.8535135135135135}
{0: 0.005431000000000001, 10: 0.0071600000000000006, 20: 0.0065390000000000005, 30: 0.00493975, 40: 0.005851000000000001, 50: 0.005863999999999999, 60: 0.00515775, 70: 0.0036959999999999996, 80: 0.0042337500000000005, 90: 0.004204, 100: 0.003969750000000001, 110: 0.004407750000000001,